## Prerequisites - GPU environment check

**Run the cell below first.** It checks the NVIDIA tools (`nvidia-smi`, `nvcc`, `nsys`, `ncu`) and the Python GPU packages (`numpy`, `numba`, `cupy`) this course needs, and prints how to fix anything missing.

- **OK** - ready to use.
- **MISSING** - a required tool; fix it before running this module.
- **optional** - only affects specific bonus paths; the workbooks skip these gracefully.

In [ ]:
# Prerequisites: verify the GPU toolchain before running this notebook.
# This finds check_gpu.py at the repository root and runs it.
import pathlib, runpy
_here = pathlib.Path.cwd()
for _d in (_here, *_here.parents):
    if (_d / "check_gpu.py").exists():
        runpy.run_path(str(_d / "check_gpu.py"), run_name="__main__")
        break
else:
    print("check_gpu.py not found - run `python check_gpu.py` from the repo root.")

# Particle-Physics Event Generation on GPUs

Event generation is one of the most CPU-expensive parts of the LHC computing budget. This notebook shows **how parts of the generation chain can already run on the GPU using existing software**, and — just as importantly — **where the open problems are, so you can contribute** to making generators GPU-friendly.

Everything here runs on a **Linux node with CVMFS**; nothing is installed locally. The generator tools (MadGraph5_aMC@NLO, PYTHIA 8, compilers, `nvcc`) come from the **Key4hep** stack:

```bash
source /cvmfs/sw.hsf.org/key4hep/setup.sh
```

The runnable code is factored into [gen-demo/gen_demo.py](gen-demo/gen_demo.py) (imported below), so the notebook cells stay short.

## The event-generation pipeline, and its GPU status

| Stage | What it does | GPU status today |
|-------|--------------|------------------|
| **Hard process** (matrix elements) | short-distance scattering amplitudes | **maturing on GPU** — MadGraph CUDACPP, MadFlow |
| **Phase-space sampling / MC integration** | pick momenta, integrate the cross section | **partly on GPU** — VegasFlow, MadFlow |
| **PDFs** | parton densities | **on GPU** — PDFFlow |
| **Parton shower** | QCD/QED radiation cascade | **mostly CPU** — an active frontier |
| **Hadronization** | strings/clusters → hadrons | **CPU** — largely open |
| **Particle decays** | e.g. EvtGen heavy-flavour decays | **CPU** — largely open |

## What you will do
1. Run **real generation on the CPU** with MadGraph5 (a `g g → t t̄` cross section).
2. Run the **same matrix element on the GPU** with MadGraph's CUDACPP backend, and compare throughput.
3. See a **map of where the GPU work is happening and where to contribute**.
4. Build intuition with a tiny **CPU-vs-GPU matrix-element** kernel (NumPy vs CuPy).


---
## 1. Environment (from CVMFS)

Source Key4hep (`source /cvmfs/sw.hsf.org/key4hep/setup.sh`) **before** launching Jupyter, so the kernel inherits `mg5_aMC`, PYTHIA 8, the compilers, and `nvcc`. The helper functions also source Key4hep in a subshell, so they work either way.

The next cell imports the factored helper module [gen-demo/gen_demo.py](gen-demo/gen_demo.py) and prints which tools are visible.


In [ ]:
import pathlib
import sys

# Import the factored helper module from ./gen-demo
sys.path.insert(0, str(pathlib.Path("gen-demo").resolve()))
import gen_demo as gd  # noqa: E402  (import after sys.path setup)

tools = gd.verify_toolchain()


---
## 2. CPU generation with MadGraph5_aMC@NLO

Let's run the **real** generator on the CPU. `gen_demo.generate_cpu()` drives the CVMFS `mg5_aMC` to compute the leading-order cross section for `g g → t t̄` (top-quark pair production) and generate parton-level events — the standard MadEvent workflow. It compiles Fortran and runs on the CPU, so it takes from tens of seconds to a few minutes.

This is the baseline: correct physics, but CPU-bound. In the next section we move the compute-heavy part — the matrix element — onto the GPU.


In [ ]:
# Real leading-order generation on the CPU (needs mg5_aMC from CVMFS/Key4hep).
# This launches MadEvent and can take a few minutes the first time.
cpu_gen = gd.generate_cpu(process="g g > t t~", ecm=13000.0, nevents=2000)


---
## 3. GPU backend: the same matrix element with CUDACPP

The cross section above is dominated by evaluating the **matrix element** at every phase-space point — millions of independent evaluations of the same formula. That is an ideal GPU workload, and it is exactly what MadGraph's **CUDACPP** backend targets ([madgraph4gpu](https://github.com/madgraph5/madgraph4gpu), bundled with MG5aMC ≥ 3.5).

`gen_demo.matrix_element_gpu_benchmark()` uses the CVMFS `mg5_aMC` to generate the **standalone CUDACPP** code for `g g → t t̄`, builds the **CPU** (scalar/SIMD) and **CUDA (GPU)** backends, and runs the built-in throughput check — reporting matrix elements per second on CPU vs GPU. Nothing is cloned or installed.

It is **detection-only by default**; pass `run=True` on a GPU node to generate, build, and benchmark (a few minutes, needs `nvcc`). Compare the `EvtsPerSec[MatrixElems]` numbers: that is where the GPU speedup shows up.


In [ ]:
# Detection only by default. On a GPU node, pass run=True to generate the CUDACPP
# standalone, build the CPU + CUDA backends, and benchmark CPU vs GPU throughput.
gd.matrix_element_gpu_benchmark(process="g g > t t~", run=False)


---
## 4. Where the GPU work is — and where you can contribute

Making event generators GPU-friendly is an active, wide-open area. Here is a map of the pipeline: what already runs on the GPU, the project that does it, and a concrete direction where new contributors can help.

| Stage | GPU project(s) | Status | Where you can contribute |
|-------|----------------|--------|--------------------------|
| Hard-process matrix elements | [madgraph4gpu / CUDACPP](https://github.com/madgraph5/madgraph4gpu) | Maturing (CUDA + SIMD; HIP/SYCL) | Add/validate processes; improve the HIP & SYCL backends; helicity/colour optimizations |
| Matrix elements (Python/ML) | [MadFlow](https://github.com/N3PDF/madflow) | Research | New processes; performance; analysis interfaces |
| MC integration & sampling | [VegasFlow](https://github.com/N3PDF/vegasflow) | Usable | Adaptive strategies; multi-GPU; variance reduction |
| Parton distribution functions | [PDFFlow](https://github.com/N3PDF/pdfflow) | Usable | More PDF sets; accuracy/perf validation |
| Parton shower | prototypes / research | **Frontier** | Port shower kernels; GPU data structures for branching |
| Hadronization (string/cluster) | — | **Largely open** | Rethink inherently sequential steps for parallel hardware |
| Particle decays (e.g. [EvtGen](https://gitlab.cern.ch/evtgen/evtgen)) | — | **Largely open** | Batched decay kinematics; amplitude evaluation on GPU |

**How to get started contributing**
- Reproduce a benchmark (e.g. the CUDACPP `check`/`gcheck` throughput for `g g → t t̄ g g`) and profile it with Nsight — a validated performance or portability fix is a great first PR.
- Pick a process not yet covered by MadFlow or CUDACPP and add it.
- The **parton shower and hadronization** stages are the biggest open problems: even prototype kernels and data-layout studies are valuable research contributions.


---
## 5. Intuition: a matrix element on CPU vs GPU (NumPy vs CuPy)

To see *why* matrix-element evaluation is such a good GPU workload — without any generator toolchain — here is a tiny stand-in. It evaluates the leading-order QED angular matrix element for `e+e- → μ+μ-` (∝ 1 + cos²θ) over millions of phase-space points on the CPU (NumPy) and, if a GPU is available, on the GPU (CuPy). Same arithmetic, many independent points: exactly the pattern the real CUDACPP kernels exploit.


In [ ]:
toy = gd.toy_matrix_element(nevents=20_000_000)


---
## 6. Summary and next steps

- **Generation is CPU-expensive**, and the compute is dominated by evaluating matrix elements over many phase-space points — a naturally data-parallel workload.
- **Existing software already runs parts on the GPU**: MadGraph's **CUDACPP** backend evaluates real matrix elements on CUDA (and SIMD CPU); **MadFlow / VegasFlow / PDFFlow** cover matrix elements, integration, and PDFs in TensorFlow.
- You ran the **CPU generation** (`g g → t t̄` with `mg5_aMC`) and the **GPU backend** (CUDACPP throughput) from the same CVMFS stack, and compared them.
- **The frontier is wide open**: parton showers, hadronization, and decays are still CPU-bound. These are real, impactful places to contribute — see the table above.

### Where to go next
- Profile the CUDACPP `check`/`gcheck` benchmark with Nsight Systems (Module 03 techniques).
- Explore [MadFlow](https://github.com/N3PDF/madflow) for a Python/TensorFlow path to GPU matrix elements.
- Read the [madgraph4gpu](https://github.com/madgraph5/madgraph4gpu) issues for "good first" contribution ideas.
